# Benchmarks: polars-uuid plugin vs. naive Python UUID generation

Compares three ways of adding a UUID column to a dataframe of `n` rows:

1. **polars + polars_uuid** — the Rust plugin's native expressions
2. **polars + naive** — `pl.Expr.map_elements` calling the stdlib `uuid` module once per row
3. **pandas + naive** — a list comprehension over the stdlib `uuid` module (the fastest common
   pandas idiom for this — faster than `Series.apply`, which pays extra per-element overhead)

Two functions are benchmarked: `uuid4` (pure random generation, no input data needed) and
`uuid5` (deterministic, derived from a string column — more representative of turning an
existing natural key into a stable UUID).

In [1]:
import timeit
import uuid

import pandas as pd
import polars as pl

import polars_uuid

SIZES = [1_000, 10_000, 100_000, 1_000_000]
REPEATS = 3  # take the minimum of this many runs, to reduce noise from other work on the machine


def timed(fn) -> float:
    """Best-of-`REPEATS` wall-clock seconds for a single call to `fn()`."""
    return min(timeit.repeat(fn, repeat=REPEATS, number=1))

## `uuid4`: random generation

In [2]:
def bench_uuid4(n: int) -> dict[str, float]:
    pldf = pl.DataFrame({"row": range(n)})
    pddf = pd.DataFrame({"row": range(n)})

    return {
        "n": n,
        "polars + polars_uuid": timed(
            lambda: pldf.with_columns(polars_uuid.uuid4("row").alias("id"))
        ),
        "polars + naive": timed(
            lambda: pldf.with_columns(
                pl.col("row")
                .map_elements(lambda _: str(uuid.uuid4()), return_dtype=pl.String)
                .alias("id")
            )
        ),
        "pandas + naive": timed(
            lambda: pddf.assign(
                id=[str(uuid.uuid4()) for _ in range(len(pddf))]
            )
        ),
    }


uuid4_results = pl.DataFrame([bench_uuid4(n) for n in SIZES])
uuid4_results

n,polars + polars_uuid,polars + naive,pandas + naive
i64,f64,f64,f64
1000,0.000598,0.005392,0.004743
10000,0.004991,0.042889,0.037824
100000,0.043514,0.39606,0.403359
1000000,0.436236,4.162669,4.018409


## `uuid5`: deterministic, derived from a string column

In [3]:
def bench_uuid5(n: int) -> dict[str, float]:
    names = [f"user-{i}@example.com" for i in range(n)]
    pldf = pl.DataFrame({"name": names})
    pddf = pd.DataFrame({"name": names})

    return {
        "n": n,
        "polars + polars_uuid": timed(
            lambda: pldf.with_columns(
                polars_uuid.uuid5("name", namespace=uuid.NAMESPACE_DNS).alias("id")
            )
        ),
        "polars + naive": timed(
            lambda: pldf.with_columns(
                pl.col("name")
                .map_elements(
                    lambda v: str(uuid.uuid5(uuid.NAMESPACE_DNS, v)),
                    return_dtype=pl.String,
                )
                .alias("id")
            )
        ),
        "pandas + naive": timed(
            lambda: pddf.assign(
                id=[str(uuid.uuid5(uuid.NAMESPACE_DNS, v)) for v in pddf["name"]]
            )
        ),
    }


uuid5_results = pl.DataFrame([bench_uuid5(n) for n in SIZES])
uuid5_results

n,polars + polars_uuid,polars + naive,pandas + naive
i64,f64,f64,f64
1000,0.000488,0.00793,0.007391
10000,0.00297,0.043363,0.047845
100000,0.020651,0.440448,0.499071
1000000,0.194311,4.589508,5.042889


## Speedup vs. naive polars, at the largest size

In [4]:
def summarize(
    results: pl.DataFrame,
    label: str,
    *,
    polars_baseline: str = "map_elements",
    pandas_baseline: str = "list comprehension",
) -> None:
    row = results.filter(pl.col("n") == SIZES[-1]).row(0, named=True)
    plugin, naive_pl, naive_pd = (
        row["polars + polars_uuid"],
        row["polars + naive"],
        row["pandas + naive"],
    )
    print(f"{label} @ n={SIZES[-1]:,}")
    print(f"  polars_uuid is {naive_pl / plugin:6.1f}x faster than naive polars ({polars_baseline})")
    print(f"  polars_uuid is {naive_pd / plugin:6.1f}x faster than naive pandas ({pandas_baseline})")


summarize(uuid4_results, "uuid4")
print()
summarize(uuid5_results, "uuid5")

uuid4 @ n=1,000,000
  polars_uuid is    9.5x faster than naive polars (map_elements)
  polars_uuid is    9.2x faster than naive pandas (list comprehension)

uuid5 @ n=1,000,000
  polars_uuid is   23.6x faster than naive polars (map_elements)
  polars_uuid is   26.0x faster than naive pandas (list comprehension)


## `uuid5` on a composite key (a more realistic workload)

Almost nobody hashes a single pre-existing text column — a real id is usually built
from several typed columns (see the README's composite-key example). This benchmark
reflects that: a 9-column key (7 integers + 2 datetimes), formatted and joined with
`concat_str` before hashing.

Unlike the single-column benchmarks above, **the type conversion and string-joining
step is identical across all three approaches** — polars vectorized formatting for
both polars variants, pandas-native formatting for the pandas variant. Only the final
hash step differs (`polars_uuid.uuid5` vs. calling `uuid.uuid5` once per row). Since
conversion is a large share of the total work and it's shared rather than
differentiating, the speedup ratio here is a lot smaller than the single-column
numbers above — that's expected, not a discrepancy, and it's the number to look at if
your workload looks like this rather than a single already-text column.

In [5]:
from datetime import datetime, timedelta

import numpy as np


def bench_uuid5_composite_key(n: int) -> dict[str, float]:
    rng = np.random.default_rng(0)
    int_cols = {f"int_{i}": rng.integers(0, 1_000_000, size=n) for i in range(7)}
    base = datetime(2024, 1, 1)
    offsets = rng.integers(0, 10_000_000, size=n)
    dt_cols = {
        f"dt_{i}": [base + timedelta(seconds=int(s)) for s in offsets + i]
        for i in range(2)
    }
    data = {**int_cols, **dt_cols}
    pldf = pl.DataFrame(data)
    pddf = pd.DataFrame(data)

    def polars_key_expr() -> pl.Expr:
        parts = [pl.col(f"int_{i}").cast(pl.Int64).cast(pl.String) for i in range(7)]
        parts += [
            pl.col(f"dt_{i}").dt.to_string("%Y-%m-%d %H:%M:%S%.6f") for i in range(2)
        ]
        return pl.concat_str(parts, separator="|")

    def pandas_key_series(df: pd.DataFrame) -> pd.Series:
        parts = [df[f"int_{i}"].astype("int64").astype(str) for i in range(7)]
        parts += [
            df[f"dt_{i}"].dt.strftime("%Y-%m-%d %H:%M:%S.%f") for i in range(2)
        ]
        out = parts[0]
        for part in parts[1:]:
            out = out.str.cat(part, sep="|")
        return out

    return {
        "n": n,
        "polars + polars_uuid": timed(
            lambda: pldf.with_columns(
                polars_uuid.uuid5(polars_key_expr(), namespace=uuid.NAMESPACE_DNS)
                .alias("id")
            )
        ),
        "polars + naive": timed(
            lambda: pldf.with_columns(polars_key_expr().alias("key")).with_columns(
                pl.col("key")
                .map_elements(
                    lambda v: str(uuid.uuid5(uuid.NAMESPACE_DNS, v)),
                    return_dtype=pl.String,
                )
                .alias("id")
            )
        ),
        "pandas + naive": timed(
            lambda: pddf.assign(
                id=pandas_key_series(pddf).apply(
                    lambda v: str(uuid.uuid5(uuid.NAMESPACE_DNS, v))
                )
            )
        ),
    }


composite_key_results = pl.DataFrame(
    [bench_uuid5_composite_key(n) for n in SIZES]
)
composite_key_results

n,polars + polars_uuid,polars + naive,pandas + naive
i64,f64,f64,f64
1000,0.001828,0.007574,0.013432
10000,0.012047,0.059083,0.108954
100000,0.10745,0.520136,1.085003
1000000,1.056776,5.444316,10.4183


In [6]:
summarize(
    composite_key_results,
    "uuid5 (9-column composite key)",
    polars_baseline="map_elements, hash step only",
    pandas_baseline=".apply(), hash step only",
)

uuid5 (9-column composite key) @ n=1,000,000
  polars_uuid is    5.2x faster than naive polars (map_elements, hash step only)
  polars_uuid is    9.9x faster than naive pandas (.apply(), hash step only)
